# FPL API Ingestion
Fetches bootstrap-static, element-summary, and fixtures from the FPL API
and writes JSON snapshots to `/Volumes/fpl/bronze/landing/`.

In [ ]:
%pip install requests --quiet

In [ ]:
import json, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
import requests

BASE_URL  = os.environ.get('FPL_API_BASE_URL', 'https://fantasy.premierleague.com/api/')
LANDING   = Path('/Volumes/fpl/bronze/landing')
TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H%M%SZ')
SESSION   = requests.Session()

def _write(sub_path, data):
    dest = LANDING / sub_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)
    print(f'Written: {dest}')

def fetch_json(url):
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_element(eid):
    try:
        return eid, fetch_json(f'{BASE_URL}element-summary/{eid}/')
    except Exception as e:
        print(f'Warning: element {eid} failed: {e}')
        return eid, None

In [ ]:
print('Fetching bootstrap-static...')
bootstrap = fetch_json(f'{BASE_URL}bootstrap-static/')
_write(f'bootstrap-static/bootstrap-static_{TIMESTAMP}.json', json.dumps(bootstrap).encode())

print('Fetching fixtures...')
_write(f'fixtures/fixtures_{TIMESTAMP}.json', json.dumps(fetch_json(f'{BASE_URL}fixtures/')).encode())

ids = [e['id'] for e in bootstrap['elements']]
print(f'Fetching {len(ids)} element summaries (threaded)...')

results = {}
with ThreadPoolExecutor(max_workers=20) as pool:
    futures = {pool.submit(fetch_element, eid): eid for eid in ids}
    for i, f in enumerate(as_completed(futures), 1):
        eid, data = f.result()
        if data:
            results[eid] = data
        if i % 100 == 0:
            print(f'  {i}/{len(ids)} done...')

payload = [{'element_id': eid, **data} for eid, data in results.items()]
_write(f'element-data/element_data_{TIMESTAMP}.json', json.dumps(payload).encode())
print(f'Done — {len(payload)} element summaries written.')